In [ ]:
import pandas as pd
import glob
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

# === Paths ===
labels_file ="\alzheimers_balanced.csv"
chunks_path = "alz_1000samples_csv_mlready/*.csv"

# === Load labels ===
labels_df = pd.read_csv(labels_file)
labels_df["sample_id"] = labels_df["sample_id"].astype(str)
labels_dict = labels_df.set_index("sample_id")["disease"].to_dict()

# Map Alzheimer's disease -> 1, others -> 0
labels_dict = {k: (1 if "Alzheimer" in str(v) else 0) for k, v in labels_dict.items()}

print(f"Labels loaded: {len(labels_dict)} samples")

# === Load chunks ===
all_data = []
for f in glob.glob(chunks_path):
    print(f"Reading {f}...")
    df = pd.read_csv(f, index_col=0)
    df.index = df.index.astype(str)
    matched = df.index.intersection(labels_df["sample_id"])
    df = df.loc[matched]
    all_data.append(df)

# Combine all CpGs
X = pd.concat(all_data, axis=1)
y = pd.Series({sid: labels_dict[sid] for sid in X.index}, name="Label")

print(f"Shape before filtering: {X.shape}")

# === Step 1: Remove zero-variance features ===
var_thresh = VarianceThreshold(threshold=0.0)
X_var = var_thresh.fit_transform(X)
X = pd.DataFrame(X_var, index=X.index, columns=X.columns[var_thresh.get_support()])

print(f"Shape after variance filter: {X.shape}")

# === Step 2: Select top-k informative CpGs ===
k = 5000  # you can change to 10000 if needed
selector = SelectKBest(score_func=f_classif, k=k)
X_new = selector.fit_transform(X, y)

# Keep selected CpG names
selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_new, index=X.index, columns=selected_features)

print(f"Shape after SelectKBest: {X.shape}")

# === Final dataset ===
final = X.copy()
final["Label"] = y

# Save efficiently
final.to_parquet("alz_mlready_top5000.parquet")
print("✅ Saved reduced dataset: alz_mlready_top5000.parquet")


Labels loaded: 1000 samples
Reading alz_1000samples_csv_mlready\mlready_chunk_1.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_10.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_11.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_12.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_13.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_14.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_15.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_16.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_17.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_18.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_19.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_2.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_20.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_21.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_22.csv...
Reading alz_1000samples_csv_mlready\mlready_chunk_23.csv...
Reading alz_10

In [ ]:
import pandas as pd
import glob
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

labels_file = "alzheimers_balanced.csv"
chunks_path = "alz_1000samples_csv_mlready/*.csv"

# === Load labels ===
labels_df = pd.read_csv(labels_file)
labels_df["sample_id"] = labels_df["sample_id"].astype(str)
labels_dict = labels_df.set_index("sample_id")["disease"].to_dict()
labels_dict = {k: (1 if "Alzheimer" in str(v) else 0) for k, v in labels_dict.items()}
print(f"Labels loaded: {len(labels_dict)} samples")

# === Load chunks ===
all_data = []
for f in glob.glob(chunks_path):
    print(f"Reading {f}...")
    df = pd.read_csv(f, index_col=0)
    df.index = df.index.astype(str)
    matched = df.index.intersection(labels_df["sample_id"])
    df = df.loc[matched]
    all_data.append(df)

X = pd.concat(all_data, axis=1)
y = pd.Series({sid: labels_dict[sid] for sid in X.index}, name="Label")

print(f"Shape before filtering: {X.shape}")

# === Step 1: quick variance filter ===
print("Applying variance filter...")
var_thresh = VarianceThreshold(threshold=0.0)
X_var = var_thresh.fit_transform(X)
X = pd.DataFrame(X_var, index=X.index, columns=X.columns[var_thresh.get_support()])
print(f"Shape after variance filter: {X.shape}")

# === Step 2: Select top-k informative CpGs ===
k = 5000  # keep only top 5000 CpGs
print(f"Selecting top {k} CpGs...")
selector = SelectKBest(score_func=f_classif, k=k)
X_new = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_new, index=X.index, columns=selected_features)
print(f"Shape after SelectKBest: {X.shape}")

# === Step 3: Save in small, fast format ===
final = X.copy()
final["Label"] = y

# Save Parquet (much faster & smaller than CSV)
out_file = "alz_mlready_top5000.parquet"
final.to_parquet(out_file, engine="pyarrow")

print(f"✅ Saved reduced dataset: {out_file}")


In [7]:
import pandas as pd

labels = pd.read_csv("alzheimers_balanced.csv")
print("Columns in labels file:", labels.columns.tolist())
print(labels.head())


Columns in labels file: ['s_no', 'sample_id', 'age', 'gender', 'sample_type', 'disease']
   s_no   sample_id   age gender     sample_type              disease
0     1  train10001  88.0      F  disease tissue  Alzheimer's disease
1     3  train10003  93.0      F  disease tissue  Alzheimer's disease
2     4  train10004  96.0      F  disease tissue  Alzheimer's disease
3     6  train10006  80.0      M  disease tissue  Alzheimer's disease
4     7  train10007  79.0      F  disease tissue  Alzheimer's disease


In [14]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import glob

# -----------------------------
# 1. Load labels
# -----------------------------
labels_file = "alzheimers_balanced.csv"
labels_df = pd.read_csv(labels_file)
labels_df["sample_id"] = labels_df["sample_id"].astype(str).str.strip()

print("Labels loaded:", labels_df.shape)

# -----------------------------
# 2. Load chunks one by one
# -----------------------------
chunk_files = glob.glob("alz_1000samples_csv_mlready/*.csv")

all_variances = []

for f in chunk_files:
    print(f"Processing {f} ...")
    df = pd.read_csv(f)

    # Ensure consistent IDs
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    df = df.merge(labels_df[["sample_id", "disease"]], on="sample_id", how="inner")

    # Keep only CpGs
    cpg_cols = [c for c in df.columns if c.startswith("cg")]
    df_cpg = df[cpg_cols].astype("float32")

    # Compute variance for this chunk
    variances = df_cpg.var(axis=0, skipna=True)
    all_variances.append(variances)

# -----------------------------
# 3. Combine variances
# -----------------------------
combined_var = pd.concat(all_variances, axis=1).mean(axis=1)

# Pick top 5000 CpGs
top_cpgs = combined_var.nlargest(50).index.tolist()
print("Selected top CpGs:", len(top_cpgs))

# -----------------------------
# 4. Reload chunks (reduced)
# -----------------------------
reduced_dfs = []

for f in chunk_files:
    df = pd.read_csv(f)
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    df = df.merge(labels_df[["sample_id", "disease"]], on="sample_id", how="inner")

    keep_cols = ["sample_id", "disease"] + [c for c in top_cpgs if c in df.columns]
    reduced_dfs.append(df[keep_cols])

# Merge all reduced chunks
df_reduced = pd.concat(reduced_dfs, axis=0)

# -----------------------------
# 5. Encode labels
# -----------------------------
le = LabelEncoder()
df_reduced["label"] = le.fit_transform(df_reduced["disease"])
df_reduced = df_reduced.drop(columns=["disease"])

print("Final shape:", df_reduced.shape)
print("Label classes:", dict(zip(le.classes_, le.transform(le.classes_))))

# -----------------------------
# 6. Save
# -----------------------------
output_file = "alz_ml_ready_reduced.csv"
df_reduced.to_csv(output_file, index=False)
print("✅ Saved:", output_file)


Labels loaded: (1000, 6)
Processing alz_1000samples_csv_mlready\mlready_chunk_1.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_10.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_11.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_12.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_13.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_14.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_15.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_16.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_17.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_18.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_19.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_2.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_20.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_21.csv ...
Processing alz_1000samples_csv_mlready\mlready_chunk_22.csv ...
Processing alz_10

In [17]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Prepare encoder first
le = LabelEncoder()
labels_df["label"] = le.fit_transform(labels_df["disease"])

# Output file paths
csv_out = "alz_ml_ready_reduced.csv"
parquet_out = "alz_ml_ready_reduced.parquet"

# Write CSV header once
header_written = False
csv_sep = ","  # Excel-friendly

# Parquet collector (list of small DataFrames)
parquet_chunks = []

for i, f in enumerate(chunk_files, 1):
    print(f"Processing chunk {i}/{len(chunk_files)} → {f}")

    df = pd.read_csv(f)
    df["sample_id"] = df["sample_id"].astype(str).str.strip()

    # Merge with labels
    df = df.merge(labels_df[["sample_id", "label"]], on="sample_id", how="inner")

    # Ensure all CpGs exist
    for cpg in top_cpgs:
        if cpg not in df.columns:
            df[cpg] = 0.0

    # Reorder
    df = df[["sample_id", "label"] + top_cpgs]

    # --- Save to CSV incrementally ---
    df.to_csv(csv_out, mode="a", header=not header_written, index=False,
              sep=csv_sep, encoding="utf-8-sig")
    header_written = True

    # --- Collect smaller parquet pieces ---
    parquet_chunks.append(df)

# Save parquet once at end (pandas will stack efficiently)
df_final = pd.concat(parquet_chunks, axis=0)
df_final.to_parquet(parquet_out, index=False)

print("✅ Done: streaming CSV + final Parquet saved.")
print("Final shape:", df_final.shape)
print("Label classes:", dict(zip(le.classes_, le.transform(le.classes_))))


Processing chunk 1/49 → alz_1000samples_csv_mlready\mlready_chunk_1.csv
Processing chunk 2/49 → alz_1000samples_csv_mlready\mlready_chunk_10.csv
Processing chunk 3/49 → alz_1000samples_csv_mlready\mlready_chunk_11.csv
Processing chunk 4/49 → alz_1000samples_csv_mlready\mlready_chunk_12.csv
Processing chunk 5/49 → alz_1000samples_csv_mlready\mlready_chunk_13.csv
Processing chunk 6/49 → alz_1000samples_csv_mlready\mlready_chunk_14.csv
Processing chunk 7/49 → alz_1000samples_csv_mlready\mlready_chunk_15.csv
Processing chunk 8/49 → alz_1000samples_csv_mlready\mlready_chunk_16.csv
Processing chunk 9/49 → alz_1000samples_csv_mlready\mlready_chunk_17.csv
Processing chunk 10/49 → alz_1000samples_csv_mlready\mlready_chunk_18.csv
Processing chunk 11/49 → alz_1000samples_csv_mlready\mlready_chunk_19.csv
Processing chunk 12/49 → alz_1000samples_csv_mlready\mlready_chunk_2.csv
Processing chunk 13/49 → alz_1000samples_csv_mlready\mlready_chunk_20.csv
Processing chunk 14/49 → alz_1000samples_csv_mlre

In [18]:
import pandas as pd

df = pd.read_parquet("alz_ml_ready_reduced.parquet")
print(df.shape)
print(df.head())


(49000, 52)
    sample_id  label  cg15265092  cg01053894  cg25103043  cg01908874  \
0  train10001      0         0.0         0.0         0.0         0.0   
1  train10003      0         0.0         0.0         0.0         0.0   
2  train10004      0         0.0         0.0         0.0         0.0   
3  train10006      0         0.0         0.0         0.0         0.0   
4  train10007      0         0.0         0.0         0.0         0.0   

   cg10530613  cg16909341  cg06242416  cg07565150  ...  cg08715988  \
0         0.0         0.0         0.0         0.0  ...         0.0   
1         0.0         0.0         0.0         0.0  ...         0.0   
2         0.0         0.0         0.0         0.0  ...         0.0   
3         0.0         0.0         0.0         0.0  ...         0.0   
4         0.0         0.0         0.0         0.0  ...         0.0   

   cg19583880  cg12482564  cg20250700  cg15451585  cg03756034  cg05084610  \
0         0.0         0.0         0.0         0.0        

In [27]:
import glob
chunk_dir = "alz_1000samples_csv_mlready"
chunk_files = sorted(glob.glob(os.path.join(chunk_dir, "mlready_chunk_*.csv")))
print("Chunk files found:", len(chunk_files))
print(chunk_files[:5])  # first 5 files


Chunk files found: 49
['alz_1000samples_csv_mlready\\mlready_chunk_1.csv', 'alz_1000samples_csv_mlready\\mlready_chunk_10.csv', 'alz_1000samples_csv_mlready\\mlready_chunk_11.csv', 'alz_1000samples_csv_mlready\\mlready_chunk_12.csv', 'alz_1000samples_csv_mlready\\mlready_chunk_13.csv']


In [32]:
import pandas as pd
import glob
import os

input_dir = "alz_1000samples_csv_mlready"
chunk_files = sorted(glob.glob(os.path.join(input_dir, "mlready_chunk_*.csv")))

for f in chunk_files:
    df = pd.read_csv(f)
    print(f"Chunk: {f}")
    print(df.columns.tolist())
    print("---")


Chunk: alz_1000samples_csv_mlready\mlready_chunk_1.csv
['sample_id', 'disease']
---
Chunk: alz_1000samples_csv_mlready\mlready_chunk_10.csv
['sample_id', 'disease', 'cg24791283', 'cg05084610']
---
Chunk: alz_1000samples_csv_mlready\mlready_chunk_11.csv
['sample_id', 'disease', 'cg14224760', 'cg11882601', 'cg26768765']
---
Chunk: alz_1000samples_csv_mlready\mlready_chunk_12.csv
['sample_id', 'cg24460369', 'cg24461063', 'cg24480012', 'cg24484802', 'cg24485820', 'cg24489813', 'cg24492884', 'cg24500832', 'cg24502904', 'cg24504927', 'cg24522654', 'cg24531568', 'cg24534098', 'cg24538626', 'cg24539569', 'cg24542852', 'cg24546205', 'cg24547869', 'cg24549639', 'cg24570064', 'cg24592370', 'cg24594830', 'cg24620454', 'cg24621371', 'cg24621412', 'cg24622544', 'cg24625634', 'cg24626079', 'cg24630778', 'cg24631609', 'cg24632597', 'cg24669694', 'cg24670383', 'cg24672547', 'cg24675150', 'cg24677744', 'cg24678505', 'cg24679411', 'cg24680632', 'cg24689794', 'cg24698875', 'cg24702147', 'cg24705741', 'cg2

In [33]:
top50_file = "alz_ml_ready_reduced.csv"
top50_df = pd.read_csv(top50_file)
top50_cpgs = [c for c in top50_df.columns if c not in ["sample_id", "label"]]
print("Top50 CpGs:", top50_cpgs)


Top50 CpGs: ['cg15265092', 'cg01445325', 'cg04247746', 'cg07943849', 'cg16908556', 'cg20250700', 'cg24791283', 'cg05084610', 'cg14224760', 'cg11882601', 'cg26768765', 'cg16909341', 'cg14569423', 'cg03552317', 'cg03039837', 'cg07589531', 'cg24030996', 'cg00562011', 'cg22052758', 'cg10617091', 'cg14041338', 'cg01908874', 'cg07565150', 'cg06746362', 'cg15451585', 'cg03312856', 'cg04918002', 'cg03715182', 'cg06242416', 'cg09250423', 'cg01053894', 'cg25103043', 'cg12114985', 'cg11461302', 'cg02129146', 'cg10530613', 'cg08715988', 'cg19583880']


In [35]:
import pandas as pd
import glob

# Your top50 CpGs
top50_cpgs = ['cg15265092', 'cg01445325', 'cg04247746', 'cg07943849', 'cg16908556', 
              'cg20250700', 'cg24791283', 'cg05084610', 'cg14224760', 'cg11882601', 
              'cg26768765', 'cg16909341', 'cg14569423', 'cg03552317', 'cg03039837', 
              'cg07589531', 'cg24030996', 'cg00562011', 'cg22052758', 'cg10617091', 
              'cg14041338', 'cg01908874', 'cg07565150', 'cg06746362', 'cg15451585', 
              'cg03312856', 'cg04918002', 'cg03715182', 'cg06242416', 'cg09250423', 
              'cg01053894', 'cg25103043', 'cg12114985', 'cg11461302', 'cg02129146', 
              'cg10530613', 'cg08715988', 'cg19583880']

# Folder containing the 49 chunk outputs with CpG values
chunk_files = sorted(glob.glob(  "alz_1000samples_csv_mlready/*.csv" ))

# Collect all columns from all chunks
all_columns = set()
for f in chunk_files:
    df = pd.read_csv(f)
    all_columns.update(df.columns)

# Check which top50 CpGs exist
existing_cpgs = [c for c in top50_cpgs if c in all_columns]
missing_cpgs = [c for c in top50_cpgs if c not in all_columns]

print("Top50 CpGs present in chunks:", existing_cpgs)
print("Top50 CpGs missing in chunks:", missing_cpgs)


Top50 CpGs present in chunks: ['cg15265092', 'cg01445325', 'cg04247746', 'cg07943849', 'cg16908556', 'cg20250700', 'cg24791283', 'cg05084610', 'cg14224760', 'cg11882601', 'cg26768765', 'cg16909341', 'cg14569423', 'cg03552317', 'cg03039837', 'cg07589531', 'cg24030996', 'cg00562011', 'cg22052758', 'cg10617091', 'cg14041338', 'cg01908874', 'cg07565150', 'cg06746362', 'cg15451585', 'cg03312856', 'cg04918002', 'cg03715182', 'cg06242416', 'cg09250423', 'cg01053894', 'cg25103043', 'cg12114985', 'cg11461302', 'cg02129146', 'cg10530613', 'cg08715988', 'cg19583880']
Top50 CpGs missing in chunks: []


In [41]:
import pandas as pd
import glob
import os

# -----------------------------
# 1. Paths (relative)
# -----------------------------
chunk_folder =  "alz_1000samples_csv_mlready"   # folder with your 49 chunk CSVs
top50_file = "alz_ml_ready_reduced.csv"                 # your top50 CpGs file
output_file = "top50_cpgs_filled.csv"       # output file

# -----------------------------
# 2. Load top50 file
# -----------------------------
top50_df = pd.read_csv(top50_file)
top50_cpgs = [c for c in top50_df.columns if c not in ["sample_id", "disease"]]
print(f"Top50 CpGs: {top50_cpgs}")

# -----------------------------
# 3. Find all chunk files
# -----------------------------
chunk_files = sorted(glob.glob(os.path.join(chunk_folder, "*.csv")))
if not chunk_files:
    raise ValueError(f"No chunk files found in {chunk_folder}")
print(f"Found {len(chunk_files)} chunk files")

# -----------------------------
# 4. Load all chunks containing top50 CpGs
# -----------------------------
chunk_dfs = []
for f in chunk_files:
    df = pd.read_csv(f)
    available_cpgs = [c for c in top50_cpgs if c in df.columns]
    if available_cpgs:
        chunk_dfs.append(df[["sample_id"] + available_cpgs])

if not chunk_dfs:
    raise ValueError("No chunks contain top50 CpGs. Cannot fill values.")

all_chunks_df = pd.concat(chunk_dfs, axis=0)
all_chunks_df = all_chunks_df.drop_duplicates(subset="sample_id")
all_chunks_df = all_chunks_df.set_index("sample_id")

# -----------------------------
# 5. Fill top50 file CpGs with actual probabilities
# -----------------------------
top50_df = top50_df.set_index("sample_id")
for cpg in top50_cpgs:
    if cpg in all_chunks_df.columns:
        top50_df[cpg] = all_chunks_df[cpg]

top50_df = top50_df.reset_index()

# -----------------------------
# 6. Save filled top50 file
# -----------------------------
top50_df.to_csv(output_file, index=False)
print(f"✅ Filled top50 CpGs saved to {output_file}")


Top50 CpGs: ['cg15265092', 'cg01445325', 'cg04247746', 'cg07943849', 'cg16908556', 'cg20250700', 'cg24791283', 'cg05084610', 'cg14224760', 'cg11882601', 'cg26768765', 'cg16909341', 'cg14569423', 'cg03552317', 'cg03039837', 'cg07589531', 'cg24030996', 'cg00562011', 'cg22052758', 'cg10617091', 'cg14041338', 'cg01908874', 'cg07565150', 'cg06746362', 'cg15451585', 'cg03312856', 'cg04918002', 'cg03715182', 'cg06242416', 'cg09250423', 'cg01053894', 'cg25103043', 'cg12114985', 'cg11461302', 'cg02129146', 'cg10530613', 'cg08715988', 'cg19583880', 'label']
Found 49 chunk files
✅ Filled top50 CpGs saved to top50_cpgs_filled.csv


In [42]:
import pandas as pd
import glob
import os

# -----------------------------
# 1. Paths
# -----------------------------
chunk_folder =  "alz_1000samples_csv_mlready"   # folder with your 49 chunk CSVs
top50_file = "alz_ml_ready_reduced.csv"                 # your top50 CpGs file
output_file = "top50_cpgs_filled_again.csv"       # final output

# -----------------------------
# 2. Load top50 CpGs file
# -----------------------------
top50_df = pd.read_csv(top50_file)
top50_cpgs = [c for c in top50_df.columns if c != "sample_id" and c != "disease"]  # CpG columns

# -----------------------------
# 3. Find all chunk files
# -----------------------------
chunk_files = sorted(glob.glob(os.path.join(chunk_folder, "*.csv")))
if not chunk_files:
    raise ValueError(f"No chunk files found in {chunk_folder}")
print(f"Found {len(chunk_files)} chunk files")

# -----------------------------
# 4. Load chunks containing top50 CpGs
# -----------------------------
chunk_dfs = []
for f in chunk_files:
    df = pd.read_csv(f)
    # Keep only columns we need
    cpgs_in_chunk = [c for c in top50_cpgs if c in df.columns]
    if cpgs_in_chunk:
        cols = ["sample_id"] + cpgs_in_chunk
        chunk_dfs.append(df[cols])

if not chunk_dfs:
    raise ValueError("No chunks contain top50 CpGs. Cannot fill values.")

# -----------------------------
# 5. Merge all chunks into one DataFrame
# -----------------------------
all_cpgs_df = pd.concat(chunk_dfs, axis=0)
all_cpgs_df = all_cpgs_df.drop_duplicates(subset="sample_id")  # ensure one row per sample

# -----------------------------
# 6. Fill top50_df CpGs from chunks
# -----------------------------
top50_df = top50_df.set_index("sample_id")
all_cpgs_df = all_cpgs_df.set_index("sample_id")

for cpg in top50_cpgs:
    if cpg in all_cpgs_df.columns:
        top50_df[cpg] = all_cpgs_df[cpg]

# Reset index and save
top50_df = top50_df.reset_index()
top50_df.to_csv(output_file, index=False)
print(f"✅ Top50 CpGs file filled and saved as {output_file}")


Found 49 chunk files
✅ Top50 CpGs file filled and saved as top50_cpgs_filled_again.csv
